In [ ]:
from __future__ import annotations
from langgraph.graph import StateGraph, START, END
import operator
from typing import TypedDict, List, Annotated
from pydantic import BaseModel, Field
from langgraph.types import Send
import os
from langchain.chat_models import init_chat_model
from langchain_core.messages import SystemMessage, HumanMessage
import psycopg
from psycopg.rows import dict_row
from langgraph.checkpoint.postgres import PostgresSaver
from dotenv import load_dotenv
from pathlib import Path

load_dotenv()

In [ ]:
DATABASE_URL = os.getenv('DATABASE_URL', '')

class Task(BaseModel):
    id: int
    title: str
    brief: str = Field(..., description='What to cover')

class Plan(BaseModel):
    blog_title: str
    tasks: List[Task]

class State(TypedDict):
    topic: str
    plan: Plan
    sections: Annotated[List[str], operator.add]

model = init_chat_model("mistralai:mistral-small-latest")

def orchestrator(state: State) -> dict:
    plan = model.with_structured_output(Plan).invoke(
        [
            SystemMessage(content="Create a multi section detailed blog plan on the following topic"),
            HumanMessage(content=f"Topic: {state['topic']}")
        ]
    )
    return {"plan": plan}

def fanout(state: State) -> list[Send]:
    return [
        Send(
            'worker',
            {
                'task': task,
                'topic': state['topic'],
                'plan': state['plan'],
            },
        )
        for task in state['plan'].tasks
    ]

def worker(payload: dict) -> dict:
    task = payload['task']
    topic = payload['topic']
    plan = payload['plan']
    blog_title = plan.blog_title

    section_md = model.invoke(
        [
            SystemMessage(content="Write one clean detailed Markdown section."),
            HumanMessage(
                content=(
                    f"Blog: {blog_title}\n"
                    f"Topic: {topic}\n\n"
                    f"Section: {task.title}\n"
                    f"Brief: {task.brief}\n\n"
                    "Return only the section content in Markdown."
                )
            ),
        ]
    ).content.strip()

    return {"sections": [section_md]}

def reducer(state: State) -> dict:
    
    title = state["plan"].blog_title
    body = "\n\n".join(state["sections"]).strip()

    final_md = f"# {title}\n\n{body}\n"

    # ---- save to file ----
    filename = title.lower().replace(" ", "_") + ".md"
    output_path = Path(filename)
    output_path.write_text(final_md, encoding="utf-8")

    return {"final": final_md}

g = StateGraph(State)
g.add_node("orchestrator", orchestrator)
g.add_node("worker", worker)
g.add_node("reducer", reducer)

g.add_edge(START, "orchestrator")
g.add_conditional_edges("orchestrator", fanout, ["worker"])
g.add_edge("worker", "reducer")
g.add_edge("reducer", END)

_conn = psycopg.connect(
    DATABASE_URL,
    autocommit=True,
    row_factory=dict_row
)

checkpointer = PostgresSaver(_conn)
checkpointer.setup()

workflow = g.compile(checkpointer=checkpointer)
workflow